In [1]:
import boto3

In [2]:
sqs = boto3.client(
    "sqs",
    endpoint_url="http://localhost:4566",
    region_name="us-east-1",
    aws_access_key_id="test",
    aws_secret_access_key="test",
)

In [3]:
queue_url = sqs.create_queue(QueueName="dataset-submission")["QueueUrl"]

In [4]:
queue_url

'http://localhost:4566/000000000000/dataset-submission'

In [7]:
import json

bucket = "data-platform-raw"

queue_arn = sqs.get_queue_attributes(
    QueueUrl=queue_url,
    AttributeNames=["QueueArn"],
)["Attributes"]["QueueArn"]

policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "s3.amazonaws.com"},
        "Action": "sqs:SendMessage",
        "Resource": queue_arn,
        "Condition": {
            "ArnEquals": {"aws:SourceArn": f"arn:aws:s3:::{bucket}"},
            "StringEquals": {"aws:SourceAccount": queue_arn.split(":")[4]},
        },
    }],
}

sqs.set_queue_attributes(
    QueueUrl=queue_url,
    Attributes={"Policy": json.dumps(policy)},
)

{'ResponseMetadata': {'RequestId': 'b8172e87-5939-4c3c-b6f8-6fa6dee80c35',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'content-type': 'application/x-amz-json-1.0;charset=UTF-8',
   'content-length': '2',
   'date': 'Wed, 04 Nov 2026 02:18:14 GMT',
   'x-amz-id-2': 'b8172e87-5939-4c3c-b6f8-6fa6dee80c35',
   'x-amz-request-id': 'b8172e87-5939-4c3c-b6f8-6fa6dee80c35',
   'x-amzn-requestid': 'b8172e87-5939-4c3c-b6f8-6fa6dee80c35'},
  'RetryAttempts': 0}}

In [9]:
s3 = boto3.client(
    "s3",
    endpoint_url="http://localhost:4566",
    region_name="us-east-1",
    aws_access_key_id="test",
    aws_secret_access_key="test",
)

In [10]:
s3.put_bucket_notification_configuration(
    Bucket=bucket,
    NotificationConfiguration={
        "QueueConfigurations": [{
            "Id": "notify-on-incoming-upload",
            "QueueArn": queue_arn,
            "Events": ["s3:ObjectCreated:*"],
            "Filter": {
                "Key": {
                    "FilterRules": [{"Name": "prefix", "Value": "incoming/"}]
                }
            },
        }]
    },
)

{'ResponseMetadata': {'RequestId': '2817de05-27cc-4cb8-bab5-6b8665f62acf',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 04 Nov 2026 02:20:13 GMT',
   'x-amz-id-2': '2817de05-27cc-4cb8-bab5-6b8665f62acf',
   'x-amz-request-id': '2817de05-27cc-4cb8-bab5-6b8665f62acf',
   'x-amzn-requestid': '2817de05-27cc-4cb8-bab5-6b8665f62acf',
   'content-length': '0'},
  'RetryAttempts': 0}}

In [12]:
sqs.get_queue_attributes(QueueUrl=queue_url, AttributeNames=["QueueArn"])

{'Attributes': {'QueueArn': 'arn:aws:sqs:us-east-1:000000000000:dataset-submission'},
 'ResponseMetadata': {'RequestId': 'ee102404-35a8-4880-820b-2b3242719330',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'content-type': 'application/x-amz-json-1.0;charset=UTF-8',
   'content-length': '83',
   'date': 'Wed, 04 Nov 2026 02:32:12 GMT',
   'x-amz-id-2': 'ee102404-35a8-4880-820b-2b3242719330',
   'x-amz-request-id': 'ee102404-35a8-4880-820b-2b3242719330',
   'x-amzn-requestid': 'ee102404-35a8-4880-820b-2b3242719330'},
  'RetryAttempts': 0}}

In [11]:
s3.get_bucket_notification(Bucket=bucket)

{'ResponseMetadata': {'RequestId': '0250ad55-3934-4935-9ed7-c31a15ae3c9b',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'content-length': '417',
   'content-type': 'application/xml',
   'date': 'Wed, 04 Nov 2026 02:30:30 GMT',
   'x-amz-id-2': '0250ad55-3934-4935-9ed7-c31a15ae3c9b',
   'x-amz-request-id': '0250ad55-3934-4935-9ed7-c31a15ae3c9b',
   'x-amzn-requestid': '0250ad55-3934-4935-9ed7-c31a15ae3c9b'},
  'RetryAttempts': 0},
 'QueueConfiguration': {'Id': 'notify-on-incoming-upload',
  'Event': 's3:ObjectCreated:*',
  'Events': ['s3:ObjectCreated:*'],
  'Queue': 'arn:aws:sqs:us-east-1:000000000000:dataset-submission'}}